In [0]:
#•	Read products.csv, customers.csv, and orders.csv into Spark DataFrames with inferSchema enabled.
productsFilePath = "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/products.csv"
display(productsFilePath)

ordersFilePath = "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/orders.csv"
display(ordersFilePath)

customersFilePath = "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/customers.csv"
display(customersFilePath)

In [0]:
productsDF = spark.read.format("csv").option("header",True).option("inferSchema",True).load(productsFilePath)
#display(productsDF)
ordersDF = spark.read.format("csv").option("header",True).option("inferSchema",True).load(ordersFilePath)
#display(ordersDF)
customersDF = spark.read.format("csv").option("header",True).option("inferSchema",True).load(customersFilePath)
#display(customersDF)

In [0]:
#•	Display the schema and row counts of all three DataFrames.
productsDF.printSchema()
customersDF.printSchema()
ordersDF.printSchema()

In [0]:
#•	Display the schema and row counts of all three DataFrames.
print("count of products details:",productsDF.count())
print("count of orders details:",ordersDF.count())
print("count of customers details:",customersDF.count())


In [0]:
#•	Save these as bronze_products_df, bronze_customers_df, and bronze_orders_df (no transformations).
bronze_products_df = productsDF
bronze_customers_df = customersDF
bronze_orders_df = ordersDF
#•	Display the schema and row counts of all three DataFrames.
print("count of products details:",bronze_products_df.count())
print("count of orders details:",bronze_orders_df.count())
print("count of customers details:",bronze_customers_df.count())
#

In [0]:
###2. Silver Layer (Cleaned & Enriched)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

In [0]:
#•	Remove duplicate order records based on ORDERID.
print("before customers details:",bronze_customers_df.count())
bronze_customers_df = bronze_customers_df.dropDuplicates(["CUSTID"])
print("after customers details:",bronze_customers_df.count())                                      

In [0]:
#•	Replace null DISCOUNT values with 0 using fillna() or coalesce().

bronze_orders_df = bronze_orders_df.fillna({"DISCOUNT":0})

#display(bronze_orders_df)


In [0]:
#•	Convert ORDERDATE to a proper date type using to_date().
bronze_orders_df1 = bronze_orders_df.withColumn("ORDERDATE",try_to_date("ORDERDATE","dd-MMM-yy"))
#display(bronze_orders_df1)

In [0]:
#•	Join orders with products to bring in PNAME, CATEGORY, and PRICE
join_orders_product_df = bronze_orders_df1.join(bronze_products_df,bronze_orders_df1.PRODID == bronze_products_df.PRODID,"inner").select("PNAME","CATEGORY","PRICE")
display(join_orders_product_df)


In [0]:
"""
join_orders_product_df = (
    bronze_orders_df.alias("o")
    .join(bronze_products_df.alias("p"), col("o.PRODID") == col("p.PRODID"), "inner")
    .withColumn("GROSS_AMOUNT", col("o.QTY") * col("p.PRICE"))
    .drop("PRODID")   # drops one of the duplicate PRODID columns
)
"""

In [0]:
#•	Add a column GROSS_AMOUNT = QTY * PRICE.


join_orders_product_df = (
    bronze_orders_df.alias("o")
    .join(bronze_products_df.alias("p"), col("o.PRODID") == col("p.PRODID"), "inner")
    .withColumn("GROSS_AMOUNT", col("o.QTY") * col("p.PRICE"))
    .select(
        col("o.ORDERID"),
        col("o.CUSTID"),
        col("o.PRODID"),
        col("o.QTY"),
        col("o.ORDERDATE"),
        col("o.DISCOUNT"),
        col("p.PNAME"),
        col("p.CATEGORY"),
        col("p.PRICE"),
        col("GROSS_AMOUNT")
    )
)

display(join_orders_product_df)



In [0]:
#•	Add a column NET_AMOUNT = GROSS_AMOUNT - DISCOUNT.
join_orders_product_df = join_orders_product_df.withColumn("NET_AMOUNT", col("GROSS_AMOUNT") - col("DISCOUNT"))
#display(join_orders_product_df)
 


In [0]:
#•	Add a column ORDER_VALUE_TIER based on NET_AMOUNT: >=2000 -> 'High', >=500 -> 'Medium', otherwise 'Low' (using when/otherwise).
join_orders_product_df = join_orders_product_df.withColumn("ORDER_VALUE_TIER", 
                                                           when(col("NET_AMOUNT") >= 2000, "High")
                                                           .when(col("NET_AMOUNT") >= 500, "Medium")
                                                           .otherwise("Low")
)
display(join_orders_product_df)


In [0]:
#•	Join the result with customers to bring in CNAME, CITY, and SEGMENT.
join_orders_product_df = (
    join_orders_product_df.alias("op")
    .join(bronze_customers_df.alias("c"), col("op.CUSTID") == col("c.CUSTID"), "inner")
    .select(
        "op.*",  # keep only needed order/product cols
        "c.CNAME", "c.CITY", "c.SEGMENT"
    )
)

 
display(join_orders_product_df)

In [0]:
#•	Add audit columns LOAD_DATE (current_date) and LOAD_TS (current_timestamp).
join_orders_product_df = join_orders_product_df.withColumn("LOAD_DATE", current_date()).withColumn("LOAD_TS", current_timestamp())
#display(join_orders_product_df)


In [0]:
 #•	Save the result as silver_orders_df.
silver_orders_df = join_orders_product_df
#display(silver_orders_df)

In [0]:
silver_orders_df.printSchema()

In [0]:
##3. Gold Layer (Business Reports)##

In [0]:
#•	gold_category_summary — group by CATEGORY: ORDER_COUNT, TOTAL_QTY, TOTAL_NET_AMOUNT, AVG_NET_AMOUNT (rounded to 2 decimals).
gold_category_summary_df = (silver_orders_df.groupBy("CATEGORY")
.agg(count("ORDERID").alias("ORDER_COUNT"), sum("QTY").alias("TOTAL_QTY"), sum("NET_AMOUNT").alias("TOTAL_NET_AMOUNT"),round(avg("NET_AMOUNT"),2).alias("AVG_NET_AMOUNT")
     )
)

display(gold_category_summary_df)

In [0]:
#•	gold_customer_summary — group by CUSTID and CNAME: ORDER_COUNT and TOTAL_SPEND (sum of NET_AMOUNT), sorted by TOTAL_SPEND descending.
gold_customer_summary_df = silver_orders_df.groupBy(silver_orders_df.CUSTID, silver_orders_df.CNAME).agg(count("ORDERID").alias("ORDER_COUNT"), sum("NET_AMOUNT").alias("TOTAL_SPEND")).orderBy(col("TOTAL_SPEND").desc())

display(gold_customer_summary_df)

In [0]:
gold_customer_summary_df = silver_orders_df.groupBy(col("CUSTID"), col("CNAME")).agg(count("ORDERID").alias("ORDER_COUNT"), sum("NET_AMOUNT").alias("TOTAL_SPEND")).orderBy(col("TOTAL_SPEND").desc())

display(gold_customer_summary_df)

In [0]:
#•	gold_top_products — using a window function (partitionBy CATEGORY, orderBy TOTAL_QTY desc, after grouping orders by PRODID/PNAME/CATEGORY), return the TOP 2 best-selling products per category with their rank.

windowspec = Window.partitionBy("CATEGORY").orderBy("TOTAL_QTY")
from pyspark.sql.functions import row_number, col
gold_top_products_df = (silver_orders_df
                        .groupBy("PNAME","PRODID","CATEGORY")
                        .agg(sum("QTY").alias("TOTAL_QTY"))
                        .withColumn("rank", rank().over(windowspec))
                        .filter(col("rank") <= 2)
                        )


display(gold_top_products_df)

##Bonus Challenge (Optional)

In [0]:
#•	Write the silver_orders_df and gold tables to Delta tables (saveAsTable) instead of just displaying them.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS MiniprojectCatalog;

In [0]:
%sql
create schema if not exists MiniprojectCatalog.MiniprojectSchema;
    


In [0]:
managed_sliver=(silver_orders_df
.write.
mode("overwrite").
format("delta")
.saveAsTable("MiniprojectCatalog.MiniprojectSchema.silver_orders")
)

managed_gold_category=(gold_category_summary_df
                      .write.mode("overwrite").
                      format("delta")
                      .saveAsTable("MiniprojectCatalog.MiniprojectSchema.gold_category_summary")
                      )
managed_gold_customer=(gold_customer_summary_df
                      .write.mode("overwrite").
                      format("delta")
                      .saveAsTable("MiniprojectCatalog.MiniprojectSchema.gold_customer_summary")
                       )
managed_gold_top_products=(gold_top_products_df
                         .write.mode("overwrite").
                         format("delta")
                         .saveAsTable("MiniprojectCatalog.MiniprojectSchema.gold_top_products")
                         )
 


In [0]:
external_sliver_df  =(silver_orders_df
                     .write.mode("overwrite")
                     .format("delta")
                     .option("path", "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/silver_orders")
                     .saveAsTable("MiniprojectCatalog.MiniprojectSchema.silver_orders_external")
                     )
external_gold_category=(gold_category_summary_df
                     .write.mode("overwrite")
                     .format("delta")
                     .option("path", "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/gold_category_summary")
                     .saveAsTable("MiniprojectCatalog.MiniprojectSchema.gold_category_summary_external"))
 
external_gold_customer_summary =(gold_customer_summary_df
                                 .write.mode("overwrite")
                                 .format("delta")
                                 .option("path", "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/gold_customer_summary")
                                 .saveAsTable("MiniprojectCatalog.MiniprojectSchema.gold_customer_summary_external")
                                 )
external_gold_top_products =(gold_top_products_df
                                 .write.mode("overwrite")
                                 .format("delta")
                                 .option("path", "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/gold_top_products_summary")
                                 .saveAsTable("MiniprojectCatalog.MiniprojectSchema.gold_top_products_external")
                                 )                                 

